In [1]:
# ============================================================================
# Gate 0 (수집 시드) + Gate 1 (어디가 2024학년도 정시/수능위주 전형결과 크롤링)
# 단일 셀. 산출물: 00_crawl_seed_university_2024.csv / 01_crawl_source_registry.csv /
#          02_admission_result_raw_2024.parquet(+csv)
# 이 셀은 "크롤링"만 담당한다 — 학과 교차매핑·모델 테이블(Gate 2 이후)은 별도 단계.
# ============================================================================
import os, re, json, time, hashlib
from pathlib import Path
from datetime import datetime

import pandas as pd
from curl_cffi import requests as creq
from bs4 import BeautifulSoup

try:
    from zoneinfo import ZoneInfo
    KST = ZoneInfo("Asia/Seoul")
except Exception:
    KST = None

# ---------------------------------------------------------------------------
# 0. 경로/설정
# ---------------------------------------------------------------------------
BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "data" / "crawl_2024_admission"
RAW_HTML_DIR = OUT_DIR / "raw_html"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_SYR = 2025          # 어디가 조회연도 (2025학년도 페이지에 2024학년도 결과가 게시됨)
RESULT_YEAR = 2024         # 실제 결과연도
SKIP_IF_CACHED = True      # 이미 저장된 raw html 있으면 재요청 생략 (재실행 시 adiga.kr 부하 방지)
REQUEST_DELAY_SEC = 0.4

def now_kst():
    return datetime.now(KST) if KST else datetime.now()

# ---------------------------------------------------------------------------
# 1. Gate 0 — 수집 대상 대학 시드 (`대학순위_상위54개_3개년.xlsx` 기준 52행,
#    단국대는 죽전/천안 두 캠퍼스로 별도 처리하여 seed 53행)
# ---------------------------------------------------------------------------
rank_df = pd.read_excel(BASE_DIR / "대학순위_상위54개_3개년.xlsx")

# 순위표 약칭 -> adiga 검색용 정식 대학명 (university_name_mapping.csv 표준명 계열과 동일 계통)
NAME_TO_SEARCH = {
    "서울대": "서울대학교", "연세대": "연세대학교", "고려대": "고려대학교", "성균관대": "성균관대학교",
    "한양대": "한양대학교", "서강대": "서강대학교", "중앙대": "중앙대학교", "이화여대": "이화여자대학교",
    "시립대": "서울시립대학교", "경희대": "경희대학교", "건국대": "건국대학교", "한국외대": "한국외국어대학교",
    "동국대": "동국대학교", "홍익대": "홍익대학교", "숭실대": "숭실대학교", "숙명여대": "숙명여자대학교",
    "국민대": "국민대학교", "부산대": "부산대학교", "세종대": "세종대학교", "아주대": "아주대학교",
    "서울과기대": "서울과학기술대학교", "경북대": "경북대학교", "인하대": "인하대학교", "성신여대": "성신여자대학교",
    "단국대": "단국대학교", "광운대": "광운대학교", "항공대": "한국항공대학교", "한양대에리카": "한양대학교",
    "상명대": "상명대학교", "가톨릭대": "가톨릭대학교", "명지대": "명지대학교", "전남대": "전남대학교",
    "인천대": "인천대학교", "가천대": "가천대학교", "덕성여대": "덕성여자대학교", "동덕여대": "동덕여자대학교",
    "경기대": "경기대학교", "한국외대(글캠)": "한국외국어대학교", "서울여대": "서울여자대학교", "충남대": "충남대학교",
    "한성대": "한성대학교", "충북대": "충북대학교", "삼육대": "삼육대학교", "서경대": "서경대학교",
    "부경대": "부경대학교", "전북대": "전북대학교", "강원대": "강원대학교", "연세대(원주캠)": "연세대학교",
    "고려대(세종캠)": "고려대학교", "수원대": "수원대학교", "한국공대": "한국공학대학교",
}

# adiga 검색은 다건(본교/분교/제N캠퍼스) 히트가 흔함 -> 정확히 어느 코드를 쓸지 명시적으로 고정.
# (군집분석/유사도로 자동선택 금지 — 개체식별 문제이므로 수동 확정)
FIXED_CODE_OVERRIDE = {
    "한양대에리카": "0000204",       # 한양대학교(ERICA)[분교]
    "연세대(원주캠)": "0000150",     # 연세대학교(미래)[분교]
    "고려대(세종캠)": "0000070",     # 고려대학교(세종)[분교]
}
BRANCH_LABEL_OVERRIDE = {
    "한양대에리카": "한양대학교(ERICA)[분교]",
    "연세대(원주캠)": "연세대학교(미래)[분교]",
    "고려대(세종캠)": "고려대학교(세종)[분교]",
}
# 순위표에 "단국대"가 2회(순위25, 47) 등장 — 죽전(본교)/천안(제2캠퍼스) 별도 캠퍼스를 의미.
DANKOOK_RANK_ORDER = {25: ("0000082", "단국대학교[본교]", "죽전"), 47: ("0002726", "단국대학교[제2캠퍼스]", "천안")}

session = creq.Session(impersonate="safari")
_home = session.get("https://www.adiga.kr/ucp/uvt/uni/univDetailSelection.do?menuId=PCUVTINF2000")
_csrf = BeautifulSoup(_home.text, "lxml").find("input", {"name": "_csrf"})["value"]

def adiga_search_univ(query, syr=SEARCH_SYR):
    data = {
        "_csrf": _csrf, "pagination.currentPage": 1, "pagination.cntPerPage": 30,
        "searchSyr": syr, "unvSeCd": 10, "sortOrder": "true", "favoriteYn": "N",
        "searchTitleInput": query, "searchTitle": query,
    }
    resp = session.post("https://www.adiga.kr/ucp/uvt/uni/univAjax.do", data=data)
    soup = BeautifulSoup(resp.text, "lxml")
    return [(a.get("code"), a.get_text(strip=True)) for a in soup.select("a.selectUniv")]

seed_rows = []
_resolve_cache = {}
for _, r in rank_df.iterrows():
    rank = int(r["순위"])
    name_raw = str(r["대학명"]).strip()
    search_query = NAME_TO_SEARCH.get(name_raw, name_raw)
    campus_note = ""
    branch_status = "main"
    manual_review = False

    if name_raw == "단국대" and rank in DANKOOK_RANK_ORDER:
        code, label, campus_note = DANKOOK_RANK_ORDER[rank]
        branch_status = "main" if rank == 25 else "branch"
    elif name_raw in FIXED_CODE_OVERRIDE:
        code, label = FIXED_CODE_OVERRIDE[name_raw], BRANCH_LABEL_OVERRIDE[name_raw]
        branch_status = "branch"
    else:
        if search_query not in _resolve_cache:
            _resolve_cache[search_query] = adiga_search_univ(search_query)
            time.sleep(REQUEST_DELAY_SEC)
        matches = _resolve_cache[search_query]
        honggyo = [m for m in matches if "[본교]" in m[1] and m[1].split("[")[0] == search_query.split("(")[0]]
        if len(honggyo) == 1:
            code, label = honggyo[0]
        elif len(matches) == 1:
            code, label = matches[0]
        else:
            code, label = (matches[0] if matches else (None, None))
            manual_review = True
            campus_note = f"검색결과 {len(matches)}건 — 자동선택 불확실: {matches}"
        branch_status = "main"

    if code is None:
        manual_review = True

    seed_rows.append({
        "univ_id": f"U{code}" if code else f"UNRESOLVED_{rank}",
        "univ_name_raw": name_raw,
        "univ_name_std": search_query,
        "campus_id": f"{code}_{campus_note}" if campus_note else code,
        "campus_name_std": campus_note or "본교/미구분",
        "adiga_univ_code": code,
        "adiga_label": label,
        "target_institution_flag": True,
        "branch_status": branch_status,
        "crawl_priority": rank,
        "manual_review_required": manual_review,
        "seed_note": campus_note if manual_review else "",
    })

seed_df = pd.DataFrame(seed_rows)

# adiga가 별도 캠퍼스코드를 제공하지 않는 경우 (예: 한국외대 글로벌캠퍼스) 동일 코드가
# 순위표의 서로 다른 두 행에 배정된다 — 군집분석/유사도로 자동합치지 않고, 먼저 나온
# 행(우선순위 rank 낮은 쪽)만 실제 크롤링 대상으로 남기고 나머지는 명시적으로 제외한다.
dup_mask = seed_df["adiga_univ_code"].notna() & seed_df["adiga_univ_code"].duplicated(keep="first")
for i in seed_df[dup_mask].index:
    kept = seed_df[(seed_df["adiga_univ_code"] == seed_df.loc[i, "adiga_univ_code"]) & (~dup_mask)]
    kept_id = kept.iloc[0]["univ_id"] if len(kept) else "?"
    seed_df.loc[i, "target_institution_flag"] = False
    seed_df.loc[i, "manual_review_required"] = True
    seed_df.loc[i, "branch_status"] = "unknown"
    seed_df.loc[i, "seed_note"] = (
        f"adiga에 별도 캠퍼스 코드 없음 — {kept_id}와 동일 페이지, 중복크롤링 방지 위해 크롤링 대상에서 제외"
    )

seed_path = OUT_DIR / "00_crawl_seed_university_2024.csv"
seed_df.to_csv(seed_path, index=False, encoding="utf-8-sig")
print(f"[Gate0] seed rows={len(seed_df)}  manual_review_required={seed_df['manual_review_required'].sum()}  "
      f"target_excluded={(~seed_df['target_institution_flag']).sum()}  -> {seed_path}")

# ---------------------------------------------------------------------------
# 2. 헤더 그리드 평탄화 (rowspan/colspan 표준 격자 알고리즘)
# ---------------------------------------------------------------------------
def flatten_header(header_trs):
    grid = {}
    max_col = 0
    for r_i, tr in enumerate(header_trs):
        col = 0
        while (r_i, col) in grid:
            col += 1
        for cell in tr.find_all(["th", "td"]):
            while (r_i, col) in grid:
                col += 1
            label = cell.get_text(" ", strip=True)
            rs = int(cell.get("rowspan", 1) or 1)
            cs = int(cell.get("colspan", 1) or 1)
            for dr in range(rs):
                for dc in range(cs):
                    grid[(r_i + dr, col + dc)] = label
            col += cs
        max_col = max(max_col, col)
    n_rows = len(header_trs)
    columns = []
    for c in range(max_col):
        parts = []
        for r_i in range(n_rows):
            v = grid.get((r_i, c), "")
            if v and (not parts or parts[-1] != v):
                parts.append(v)
        columns.append(" / ".join(parts) if parts else f"col{c}")
    return columns

def map_semantic_fields(columns, cell_texts):
    """헤더 키워드 기반 best-effort 시맨틱 매핑. 실패해도 raw_cells_json은 그대로 보존."""
    out = {k: None for k in [
        "raw_admission_group", "raw_recruitment_unit", "raw_recruitment_n",
        "raw_competition_rate", "raw_additional_rank", "raw_score_70cut",
        "raw_score_max", "raw_percentile_70cut",
    ]}
    used_pct = used_cut = False
    for col_label, val in zip(columns, cell_texts):
        c = col_label.replace(" ", "")
        if "구분" in c and out["raw_admission_group"] is None and ("군" in val or val == ""):
            out["raw_admission_group"] = val
        elif "모집단위" in c:
            out["raw_recruitment_unit"] = val
        elif ("모집" in c and "인원" in c):
            out["raw_recruitment_n"] = val
        elif "경쟁률" in c:
            out["raw_competition_rate"] = val
        elif "충원" in c:
            out["raw_additional_rank"] = val
        elif "백분위" in c and not used_pct:
            out["raw_percentile_70cut"] = val
            used_pct = True
        elif "총점" in c:
            out["raw_score_max"] = val
        elif ("cut" in c.lower() or "환산" in c) and not used_cut:
            out["raw_score_70cut"] = val
            used_cut = True
    return out

# ---------------------------------------------------------------------------
# 3. Gate 1 — 어디가 페이지 수집 + Ⅳ.수능위주전형 탭 내 정시 결과 표 파싱
# ---------------------------------------------------------------------------
registry_rows = []
raw_rows = []
_source_seq = 0

targets = seed_df[seed_df["adiga_univ_code"].notna() & seed_df["target_institution_flag"]].reset_index(drop=True)

for _, seed in targets.iterrows():
    code = seed["adiga_univ_code"]
    univ_id = seed["univ_id"]
    html_path = RAW_HTML_DIR / f"{code}_{SEARCH_SYR}.html"
    url = (f"https://www.adiga.kr/ucp/uvt/uni/univDetailSelection.do"
           f"?menuId=PCUVTINF2000&searchSyr={SEARCH_SYR}&unvCd={code}")

    if SKIP_IF_CACHED and html_path.exists():
        html = html_path.read_text(encoding="utf-8")
        http_status = 200
    else:
        resp = session.get(url)
        http_status = resp.status_code
        html = resp.text if http_status == 200 else ""
        if html:
            html_path.write_text(html, encoding="utf-8")
        time.sleep(REQUEST_DELAY_SEC)

    _source_seq += 1
    source_id = f"SRC_{code}_{SEARCH_SYR}_{_source_seq:04d}"
    registry_rows.append({
        "source_id": source_id,
        "crawl_run_id": f"RUN_{now_kst():%Y%m%d}",
        "univ_id": univ_id,
        "campus_id": seed["campus_id"],
        "source_type": "adiga_html",
        "source_priority": 1,
        "source_search_year": SEARCH_SYR,
        "source_result_year": RESULT_YEAR,
        "source_section": "Ⅳ. 수능위주전형 / Q2. 2024학년도 전형 결과",
        "source_url": url,
        "retrieved_at": now_kst().isoformat(),
        "http_status": http_status,
        "content_type": "text/html",
        "content_sha256": hashlib.sha256(html.encode("utf-8")).hexdigest() if html else "",
        "raw_file_path": str(html_path.relative_to(BASE_DIR)) if html else "",
        "parser_version": "adiga_v1",
        "parse_status": "pending",
        "parse_note": "",
    })

    if not html:
        registry_rows[-1]["parse_status"] = "failed"
        registry_rows[-1]["parse_note"] = f"http_status={http_status}"
        continue

    soup = BeautifulSoup(html, "lxml")
    tab_containers = soup.select(".tabCon.univInfoCon")
    csat_container = tab_containers[3] if len(tab_containers) >= 4 else None
    if csat_container is None:
        registry_rows[-1]["parse_status"] = "failed"
        registry_rows[-1]["parse_note"] = f"Ⅳ탭 컨테이너 미검출 (tabCon 개수={len(tab_containers)})"
        continue

    tables = csat_container.find_all("table")
    n_result_tables = 0
    for t_i, table in enumerate(tables):
        table_text = table.get_text(" ", strip=True)
        has_result_signature = (
            "모집단위" in table_text and "경쟁률" in table_text and "충원" in table_text
            and ("cut" in table_text.lower() or "백분위" in table_text)
        )
        if not has_result_signature:
            continue  # 결과표가 아닌 설명/반영비율/산출방법 표는 건너뜀

        rows = table.find_all("tr")
        if not rows:
            continue

        row0_cells = rows[0].find_all(["td", "th"])
        if len(row0_cells) == 1:
            selection_name = row0_cells[0].get_text(" ", strip=True)
            idx = 1
        else:
            selection_name = None
            idx = 0

        MAX_HEADER_ROWS = 3  # 실제 결과표 헤더는 1~2행 rowspan 구조가 전부 — 초과 시 후보 아님으로 판단
        header_trs = []
        while idx < len(rows) and len(header_trs) < MAX_HEADER_ROWS:
            cells = rows[idx].find_all(["td", "th"])
            texts = [c.get_text(" ", strip=True) for c in cells]
            row_text = " ".join(texts)
            has_digit_heavy = bool(re.search(r"\d+(\.\d+)?\s*(:1|%|명)?$", texts[-1])) if texts else False
            looks_header = any(k in row_text for k in
                                ["모집단위", "구분", "모집 인원", "경쟁률", "충원", "cut", "백분위", "환산", "총점"])
            if not header_trs and looks_header:
                header_trs.append(rows[idx]); idx += 1
                continue
            if header_trs and not has_digit_heavy and len(texts) <= 4 and looks_header:
                header_trs.append(rows[idx]); idx += 1
                continue
            break

        if not header_trs or idx >= len(rows):
            continue
        _flat_check = flatten_header(header_trs)
        if len(_flat_check) > 15:
            # 헤더 격자가 비정상적으로 커지면(설명형 표 오탐) 스킵 — raw_file_path로 수동 확인 대상
            continue

        columns = flatten_header(header_trs)
        n_result_tables += 1
        prev_values = {}
        for r_i, tr in enumerate(rows[idx:]):
            cells = tr.find_all(["td", "th"])
            cell_texts = [c.get_text(" ", strip=True) for c in cells]
            if not any(cell_texts):
                continue

            deficit = len(columns) - len(cell_texts)
            parse_warning = ""
            if deficit > 0:
                carried = [prev_values.get(i, "") for i in range(deficit)]
                cell_texts = carried + cell_texts
                parse_warning = f"{deficit}개 선두 컬럼 rowspan 승계"
            elif deficit < 0:
                parse_warning = f"컬럼수 불일치 header={len(columns)} data={len(cell_texts)+(-deficit)}"
                cell_texts = cell_texts[: len(columns)]

            for i, v in enumerate(cell_texts):
                prev_values[i] = v

            semantic = map_semantic_fields(columns, cell_texts)
            raw_rows.append({
                "raw_row_id": f"{source_id}_T{t_i}_R{r_i}",
                "source_id": source_id,
                "univ_id": univ_id,
                "admission_year": RESULT_YEAR,
                "raw_table_index": t_i,
                "raw_row_index": r_i,
                "raw_section_title": selection_name,
                "raw_header_json": json.dumps(columns, ensure_ascii=False),
                "raw_cells_json": json.dumps(cell_texts, ensure_ascii=False),
                **semantic,
                "raw_parse_warning": parse_warning,
            })

    registry_rows[-1]["parse_status"] = "success" if n_result_tables else "partial"
    registry_rows[-1]["parse_note"] = f"result_tables={n_result_tables}"

registry_df = pd.DataFrame(registry_rows)
raw_df = pd.DataFrame(raw_rows)

registry_path = OUT_DIR / "01_crawl_source_registry.csv"
registry_df.to_csv(registry_path, index=False, encoding="utf-8-sig")

raw_parquet_path = OUT_DIR / "02_admission_result_raw_2024.parquet"
raw_csv_path = OUT_DIR / "02_admission_result_raw_2024.csv"
raw_df.to_parquet(raw_parquet_path, index=False)
raw_df.to_csv(raw_csv_path, index=False, encoding="utf-8-sig")

print(f"[Gate1] sources fetched={len(registry_df)}  raw rows={len(raw_df)}")
print(f"        success={sum(registry_df['parse_status']=='success')}  "
      f"partial={sum(registry_df['parse_status']=='partial')}  "
      f"failed={sum(registry_df['parse_status']=='failed')}")
print(f"        -> {registry_path}")
print(f"        -> {raw_parquet_path} / {raw_csv_path}")

_failed = registry_df[registry_df["parse_status"] == "failed"]
if len(_failed):
    print("\n[주의] 파싱 실패 대학 (raw_file_path 확인 후 수동 점검 필요):")
    print(_failed[["univ_id", "http_status", "parse_note"]].to_string(index=False))


[Gate0] seed rows=52  manual_review_required=1  target_excluded=1  -> /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/00_crawl_seed_university_2024.csv


[Gate1] sources fetched=51  raw rows=3096
        success=51  partial=0  failed=0
        -> /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/01_crawl_source_registry.csv
        -> /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/02_admission_result_raw_2024.parquet / /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/02_admission_result_raw_2024.csv
